In [15]:
import pandas as pd
import numpy as np

In [18]:
df = pd.read_csv('pd_concat_aggregated_results_performance.csv')
df.drop(columns=['Unnamed: 0'], inplace=True)
df

,Range,A,B,C,D,E,A + B,RMSE,MSE,MAE,MAPE
0,ENTIRE,87.20,11.79,0.03,0.98,0.00,98.99,21.142341,446.998596,14.927038,0.101284
1,TBR_2,94.80,0.00,0.00,5.17,0.03,94.80,11.993165,143.835999,7.815367,0.161907
2,TBR_1,78.41,0.00,0.00,21.58,0.02,78.41,14.881750,221.466492,10.048975,0.161883
3,TIR,83.58,16.39,0.03,0.00,0.00,99.97,19.637016,385.612396,13.754208,0.112865
4,TAR_1,92.32,7.50,0.02,0.15,0.01,99.82,23.054079,531.490540,16.525259,0.078700
5,TAR_2,96.72,3.07,0.01,0.20,0.00,99.79,25.944796,673.132446,19.570175,0.065056
6,ENTIRE,86.05,12.82,0.03,1.10,0.00,98.87,21.485838,461.641235,15.230930,0.105782
7,TBR_2,94.35,0.00,0.00,5.64,0.01,94.35,11.780566,138.781738,8.052630,0.166868
8,TBR_1,76.80,0.00,0.00,23.19,0.01,76.80,15.164257,229.954697,10.371614,0.166861
9,TIR,82.66,17.31,0.03,0.00,0.00,99.97,19.977874,399.115448,14.096351,0.116443


In [19]:
logical_order = ['ENTIRE', 'TBR_2', 'TBR_1', 'TIR', 'TAR_1', 'TAR_2']

# Select only numeric columns for calculation
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# ---------------------------------------------------------
# APPROACH A: Flat CSV Structure (Numeric)
# Best for saving to CSV and further analysis
# ---------------------------------------------------------

# 1. Group and calculate Mean and Std
df_grouped = df.groupby('Range')[numeric_cols].agg(['mean', 'std'])

# 2. FLATTEN THE COLUMNS (The key step)
# This changes MultiIndex ('A', 'mean') to single index 'A_Mean'
df_grouped.columns = [f"{col}_{stat.capitalize()}" for col, stat in df_grouped.columns]

# 3. Reorder rows logically and reset index to make 'Range' a normal column
df_flat = df_grouped.reindex(logical_order).reset_index()

# print("\n=== OPTION A: Flat Structure (Ready for CSV) ===")

# To save:
# df_flat.to_csv('statistical_analysis.csv', index=False)

# ---------------------------------------------------------
# APPROACH B: Report Structure (Text format "Mean ± Std")
# Best for presentations/publications
# ---------------------------------------------------------

df_report = pd.DataFrame(index=logical_order)
df_report.index.name = 'Range'

# Calculate base stats again (or reuse from above)
grouped = df.groupby('Range')[numeric_cols]
means = grouped.mean()
stds = grouped.std()

# Format columns
for col in numeric_cols:
    df_report[col] = (
        means[col].map('{:.2f}'.format) + 
        " ± " + 
        stds[col].map('{:.2f}'.format)
    )

df_report = df_report.reset_index()

# print("\n=== OPTION B: Report Structure (Formatted) ===")

# To save:
# df_report.to_csv('final_report.csv', index=False)


=== OPTION A: Flat Structure (Ready for CSV) ===

=== OPTION B: Report Structure (Formatted) ===


In [20]:
df_flat

,Range,A_Mean,A_Std,B_Mean,B_Std,C_Mean,C_Std,D_Mean,D_Std,E_Mean,...,A + B_Mean,A + B_Std,RMSE_Mean,RMSE_Std,MSE_Mean,MSE_Std,MAE_Mean,MAE_Std,MAPE_Mean,MAPE_Std
0,ENTIRE,86.302,0.620741,12.700,0.613148,0.030,0.000000,0.966,0.102372,0.004,...,99.002,0.104738,21.806517,0.680332,475.894489,30.030798,15.328811,0.403579,0.104351,0.002444
1,TBR_2,94.272,0.980036,0.000,0.000000,0.000,0.000000,5.708,0.981208,0.020,...,94.272,0.980036,12.235628,0.905097,150.365964,23.206706,7.946517,0.414294,0.164884,0.007680
2,TBR_1,78.148,1.327618,0.000,0.000000,0.000,0.000000,21.834,1.322585,0.020,...,78.148,1.327618,15.498808,1.017319,241.041010,32.858963,10.286682,0.538714,0.165443,0.008885
3,TIR,82.850,0.583438,17.114,0.576394,0.034,0.005477,0.000,0.000000,0.000,...,99.964,0.008944,20.217322,0.563856,408.994452,22.907836,14.100939,0.317876,0.115293,0.002308
4,TAR_1,91.410,0.847673,8.368,0.797728,0.020,0.000000,0.190,0.046368,0.014,...,99.778,0.050695,24.000543,0.913970,576.694361,44.612822,17.162483,0.604196,0.081822,0.002872
5,TAR_2,95.994,0.576611,3.704,0.489060,0.010,0.000000,0.292,0.084676,0.000,...,99.698,0.088994,26.876790,0.914851,723.031421,49.919843,20.257388,0.690704,0.067944,0.002386


In [21]:
df_report

,Range,A,B,C,D,E,A + B,RMSE,MSE,MAE,MAPE
0,ENTIRE,86.30 ± 0.62,12.70 ± 0.61,0.03 ± 0.00,0.97 ± 0.10,0.00 ± 0.01,99.00 ± 0.10,21.81 ± 0.68,475.89 ± 30.03,15.33 ± 0.40,0.10 ± 0.00
1,TBR_2,94.27 ± 0.98,0.00 ± 0.00,0.00 ± 0.00,5.71 ± 0.98,0.02 ± 0.01,94.27 ± 0.98,12.24 ± 0.91,150.37 ± 23.21,7.95 ± 0.41,0.16 ± 0.01
2,TBR_1,78.15 ± 1.33,0.00 ± 0.00,0.00 ± 0.00,21.83 ± 1.32,0.02 ± 0.01,78.15 ± 1.33,15.50 ± 1.02,241.04 ± 32.86,10.29 ± 0.54,0.17 ± 0.01
3,TIR,82.85 ± 0.58,17.11 ± 0.58,0.03 ± 0.01,0.00 ± 0.00,0.00 ± 0.00,99.96 ± 0.01,20.22 ± 0.56,408.99 ± 22.91,14.10 ± 0.32,0.12 ± 0.00
4,TAR_1,91.41 ± 0.85,8.37 ± 0.80,0.02 ± 0.00,0.19 ± 0.05,0.01 ± 0.01,99.78 ± 0.05,24.00 ± 0.91,576.69 ± 44.61,17.16 ± 0.60,0.08 ± 0.00
5,TAR_2,95.99 ± 0.58,3.70 ± 0.49,0.01 ± 0.00,0.29 ± 0.08,0.00 ± 0.00,99.70 ± 0.09,26.88 ± 0.91,723.03 ± 49.92,20.26 ± 0.69,0.07 ± 0.00


In [7]:
logic_order = ['ENTIRE', 'TBR_2', 'TBR_1', 'TIR', 'TAR_1', 'TAR_2']

In [13]:
# 1. Agrupamos por 'Range' y calculamos media y desviación
# numeric_only=True evita errores con columnas de texto no deseadas
resumen_tecnico = df.groupby('Range')[['A + B', 'RMSE', 'MAE']].agg(['mean', 'std'])

# 2. Reordenamos las filas según tu orden lógico
resumen_tecnico = resumen_tecnico.reindex(logic_order)

print("--- Estructura MultiIndex (Mejor para seguir calculando) ---")
resumen_tecnico.to_csv('resumen_tecnico_multiindex.csv', index=False)

--- Estructura MultiIndex (Mejor para seguir calculando) ---


In [14]:
# Creamos una copia para no alterar el original
resumen_reporte = pd.DataFrame(index=logic_order)

# Lista de columnas numéricas que queremos analizar
cols_numericas = ['A + B', 'RMSE', 'MAE']

# Agrupamos una vez para obtener los datos
grupos = df.groupby('Range')[cols_numericas]
medias = grupos.mean()
desviaciones = grupos.std()

# Iteramos para crear el formato "X.XX ± Y.YY"
for col in cols_numericas:
    # round(2) redondea a 2 decimales
    # astype(str) convierte a texto
    resumen_reporte[col] = (
        medias[col].round(2).astype(str) + 
        " ± " + 
        desviaciones[col].round(2).astype(str)
    )

print("--- Estructura para Reporte (Mejor lectura humana) ---")
resumen_reporte

--- Estructura para Reporte (Mejor lectura humana) ---


,A + B,RMSE,MAE
ENTIRE,99.0 ± 0.1,21.81 ± 0.68,15.33 ± 0.4
TBR_2,94.27 ± 0.98,12.24 ± 0.91,7.95 ± 0.41
TBR_1,78.15 ± 1.33,15.5 ± 1.02,10.29 ± 0.54
TIR,99.96 ± 0.01,20.22 ± 0.56,14.1 ± 0.32
TAR_1,99.78 ± 0.05,24.0 ± 0.91,17.16 ± 0.6
TAR_2,99.7 ± 0.09,26.88 ± 0.91,20.26 ± 0.69


In [5]:
df_grouped_ave = df.groupby(['Range']).mean().reset_index()
df_grouped_ave

,Range,Unnamed: 0,A,B,C,D,E,A + B,RMSE,MSE,MAE,MAPE
0,ENTIRE,0.0,86.302,12.700,0.030,0.966,0.004,99.002,21.806517,475.894489,15.328811,0.104351
1,TAR_1,4.0,91.410,8.368,0.020,0.190,0.014,99.778,24.000543,576.694361,17.162483,0.081822
2,TAR_2,5.0,95.994,3.704,0.010,0.292,0.000,99.698,26.876790,723.031421,20.257388,0.067944
3,TBR_1,2.0,78.148,0.000,0.000,21.834,0.020,78.148,15.498808,241.041010,10.286682,0.165443
4,TBR_2,1.0,94.272,0.000,0.000,5.708,0.020,94.272,12.235628,150.365964,7.946517,0.164884
5,TIR,3.0,82.850,17.114,0.034,0.000,0.000,99.964,20.217322,408.994452,14.100939,0.115293


In [6]:
df_grouped_std = df.groupby(['Range']).std().reset_index()
df_grouped_std

,Range,Unnamed: 0,A,B,C,D,E,A + B,RMSE,MSE,MAE,MAPE
0,ENTIRE,0.0,0.620741,0.613148,0.000000,0.102372,0.005477,0.104738,0.680332,30.030798,0.403579,0.002444
1,TAR_1,0.0,0.847673,0.797728,0.000000,0.046368,0.005477,0.050695,0.913970,44.612822,0.604196,0.002872
2,TAR_2,0.0,0.576611,0.489060,0.000000,0.084676,0.000000,0.088994,0.914851,49.919843,0.690704,0.002386
3,TBR_1,0.0,1.327618,0.000000,0.000000,1.322585,0.007071,1.327618,1.017319,32.858963,0.538714,0.008885
4,TBR_2,0.0,0.980036,0.000000,0.000000,0.981208,0.007071,0.980036,0.905097,23.206706,0.414294,0.007680
5,TIR,0.0,0.583438,0.576394,0.005477,0.000000,0.000000,0.008944,0.563856,22.907836,0.317876,0.002308
